# Chapter 7 — Elementwise Arithmetic

Part 3: Small mathematical operations.

**Ladder:** Python element-by-element reference (Method 01) → one Triton micro-kernel per
operation (Method 02) → fused arithmetic chain and production-shaped kernels (Method 03).

**Study tip.** Fix one tiny shape for the whole chapter: $N=6$, `BLOCK=4`, so
$P=\lceil 6/4\rceil=2$ programs and the second tile is half empty. Write the tile map
before reading any kernel.

| pid | offsets | mask | live lanes |
|---|---|---|---|
| 0 | 0,1,2,3 | T,T,T,T | 0,1,2,3 |
| 1 | 4,5,6,7 | T,T,F,F | 4,5 |

- **Mask rule.** The two masked lanes are why every load and store carries a mask; the
  arithmetic never changes, only the address range.
- **Read.** An elementwise kernel is a scalar function lifted to a tile.
- **Write.** The tile map, mask, and pointer offsets for a partial last block.
- **Choose.** The right operator for a family, and state its domain.
- **Explain.** Why elementwise arithmetic is memory-bound.
- **Select.** `other` from the operation that follows, not by habit.
- **Derive.** The backward pass of the five families and check it with autograd.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Chapter front matter

**Connection.** Chapter 6 supplied pointers, offsets, `tl.load`, `tl.store`, and masking;
this chapter applies them to the smallest useful kernel.

[Index](../../00_index.ipynb) · [← Previous](../../part2_memory/ch06_indexing/06_complete.ipynb) · [Next →](../ch08_comparisons/08_complete.ipynb)

- **Estimated time.** 75–90 min.
- **Difficulty.** Intermediate.
- **Prerequisites.** Chapter 6 (pointers, offsets, loads, stores, masking); basic calculus
  (chain rule, power rule). No prior GPU arithmetic background is assumed.

## Chapter map

| § | Topic | Key formula | Applied kernel |
|---|---|---|---|
| **7.1** | Addition | $z_i=x_i+y_i$ | Affine / residual |
| **7.2** | Subtraction | $z_i=x_i-y_i$ | Affine / residual |
| **7.3** | Multiplication | $z_i=x_i\,y_i$ | Affine / residual |
| **7.4** | Division | $z_i=x_i/y_i$ | Affine / residual |
| **7.5** | Floor division | $z_i=\lfloor x_i/y_i\rfloor$ | — |
| **7.6** | Modulo | $z_i=x_i-y_i\lfloor x_i/y_i\rfloor$ | — |
| **7.7** | Power | $z_i=x_i^{\,y_i}$ | — |
| **7.8** | Negation | $z_i=-x_i$ | — |
| **7.9** | Absolute value | $z_i=\lvert x_i\rvert$ | — |
| **7.10** | Reciprocal | $z_i=1/x_i$ | — |
| **7.11** | Square | $z_i=x_i^2$ | RMS normalization |
| **7.12** | Square root | $z_i=\sqrt{x_i}$ | RMS normalization |
| **7.13** | Reciprocal square root | $z_i=1/\sqrt{x_i}$ | RMS normalization |
| **7.14** | Exponential | $z_i=e^{x_i}$ | Stable log-sum-exp |
| **7.15** | Natural logarithm | $z_i=\ln x_i$ | Stable log-sum-exp |
| **7.16** | Base-2 exponential | $z_i=2^{x_i}$ | — |
| **7.17** | Base-2 logarithm | $z_i=\log_2 x_i$ | — |
| **7.18** | Maximum | $z_i=\max(x_i,y_i)$ | Log-sum-exp · bounds |
| **7.19** | Minimum | $z_i=\min(x_i,y_i)$ | Bounds |
| **7.20** | Clamping | $z_i=\min(\max(x_i,l),u)$ | Bounds |
| **7.21** | The canonical elementwise kernel | one skeleton, `OP: tl.constexpr` | bridge to Part IV |
| **★** | Capstone | stable softmax | all five families |

## Concept diagrams

- **Elementwise pipeline.** One output lane per input lane; no lane talks to another lane.

```text
load tile x ──► f(x) ──► store tile z
              one output
              per input
```

- **Tiled launch.** Program `pid` owns the contiguous tile `[pid*BLOCK, (pid+1)*BLOCK)`.

```text
global index:  0  1  2  3 | 4  5  6  7
tile:          [ pid 0   ] | [ pid 1  ]
mask:           T  T  T  T | T  T  F  F
live output:   z0 z1 z2 z3 | z4 z5
```

**Bridge.** Part I fixes the contract, the spine, the cost model, and the five families
before any kernel appears.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import math
import pytest
import torch
import triton
import triton.language as tl

from ttools import require_gpu, run_if_gpu


def _active_device():
    """Pick Triton's active device, else CUDA, else CPU (study helper)."""
    try:
        return triton.runtime.driver.active.get_active_torch_device()
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


## Part I — Mathematics

**Thread.** The chapter is one chain; each section names the earlier link it uses.

- **Elementwise lift.** A scalar $f$ acts on lanes independently: $F(x)_i=f(x_i)$.
- **Tile skeleton.** `pid`, `offs`, `mask`, `load`, `store` are identical for all twenty
  operations.
- **Memory-bound.** Traffic, not FLOPs, sets the lower bound; fusion and narrow dtypes are
  the levers.
- **Five families.** Field, quotient/remainder, powers/roots, transcendentals,
  order/bounds each share a reason and a failure mode.
- **Backward.** Every local derivative is another elementwise kernel; a broadcast axis
  turns some gradients into reductions.

### 1. The elementwise contract

**Connection.** Chapter 6 supplied pointers, offsets, and masks; this section fixes the
map those pieces implement.

- **Rule.** For $f:\mathbb R\to\mathbb R$, the lift is $F(x)_i=f(x_i)$ for
  $i\in[N]=\{0,\dots,N-1\}$.
- **Formula.** Binary lift: $F(x,y)_i=f(x_i,y_i)$.
- **Example.** $f(t)=\max(t,0)$ gives ReLU lane by lane; $f(a,b)=a+b$ gives $z_i=x_i+y_i$.
- **Where.** Every kernel in the chapter: field, integer, root, transcendental, and bound
  operations.
- **Triton.** The scalar body is $f$; `tl.load` and `tl.store` supply the lift.
- **Failure.** An operation that reads another lane is not elementwise; it is a reduction
  (Chapter 14). Broadcasting and masking are not new operations either: a size-1 axis is
  stride-0 replication, and a mask is the lift restricted to existing indices.
- **Bridge.** §2 fixes one tiny shape so every later cell can be checked by hand.

### 2. The chapter spine

**Connection.** §1 defined the lift; this section instantiates it on the single shape used
by every kernel.

- **Rule.** $N=6$, $B=4$, $P=2$; $x=(1,2,3,4,5,6)$ and $y=(6,5,4,3,2,1)$.
- **Formula.** Program $p$ owns $I_p=\{pB,\dots,pB+B-1\}\cap[N]$; the mask is
  $m_i=[i\in I_p]$.
- **Example.** Program 0 owns $0,1,2,3$; program 1 owns $4,5,6,7$ but only $4,5$ are live.

| pid | offsets | mask | live lanes | $x$ tile (`other=0`) | $y$ tile |
|---|---|---|---|---|---|
| 0 | 0,1,2,3 | T,T,T,T | 0,1,2,3 | 1,2,3,4 | 6,5,4,3 |
| 1 | 4,5,6,7 | T,T,F,F | 4,5 | 5,6,0,0 | 2,1,0,0 |

- **Where.** Every load, store, test, and worked example in Part III.
- **Triton.** `offs = pid*BLOCK + tl.arange(0, BLOCK)`; `mask = offs < N`.
- **Failure.** Lanes 6 and 7 are computed but not stored; an unmasked store would write
  out of bounds.
- **Bridge.** §3 prices this tile and shows why the mask changes the address range, not
  the cost model.

### 3. Why arithmetic is memory-bound

**Connection.** §2 fixed the tile; this section prices the bytes that tile must move.

- **Rule.** For $N$ elements of $b$ bytes, a unary map moves $Q_{\text{unary}}=2Nb$ and a
  binary map moves $Q_{\text{binary}}=3Nb$.
- **Formula.** $\mathrm{AI}_{\text{unary}}=1/(2b)$ and
  $\mathrm{AI}_{\text{binary}}=1/(3b)$; the roofline is
  $t\ge\max(N/P_{\text{peak}},Q/\mathrm{BW})\approx Q/\mathrm{BW}$.
- **Example.** For `float32`, a binary map has $\mathrm{AI}\approx0.083$ FLOP/byte while a
  modern accelerator has a machine balance of roughly $10$–$100$ FLOP/byte. Report
  achieved bandwidth as $Q/(t\cdot10^9)$ GB/s.
- **Where.** The lever is fewer bytes: fusion (Chapter 20), narrower dtypes (Chapter 12),
  coalesced access (Chapter 13).
- **Triton.** One load per operand and one store per output; arithmetic in registers is
  nearly free.
- **Failure.** Counting FLOPs instead of bytes optimizes the wrong side of the roofline.
- **Bridge.** §4 states the floating-point rules that decide whether the few FLOPs are
  correct.

### 4. The floating-point contract

**Connection.** §3 priced bytes; this section states what those bytes mean and where they
break.

- **Rule.** Elementwise operations preserve order: lane $i$ is computed by itself, with no
  reassociation (reductions are Chapter 14).
- **Formula.** `inf`, `-inf`, `nan`, and `-0.0` are values of the type; a finite nonzero
  numerator over zero is $\pm\infty$, and $0/0$ is `nan`.
- **Example.** Numeric formats used in the chapter:

| Format | Significand bits | Decimal digits | Typical use |
|---|---:|---:|---|
| `float32` | 24 | ~7 | default arithmetic and reductions |
| `tf32` | 11 | ~3.3 | tensor-core inputs |
| `float16` | 11 | ~3.3 | storage and fast tensor-core math |
| `bfloat16` | 8 | ~2.4 | storage with a wide exponent range |

- **Where.** Storage may be narrow; compute in `float32` unless the range is proven safe.
- **Triton.** The dtype of the load decides the arithmetic; cast once at the store.
- **Failure.** Domain is an API: `log(x)` needs $x\gt 0$, `sqrt(x)` needs $x\ge0$, `rsqrt(x)`
  needs $x\gt 0$, and a real power with a negative base needs an integer exponent. The kernel
  returns `nan` or `inf` rather than stopping.
- **Bridge.** §5 groups the twenty operations by the reason they appear.

### 5. The five families

**Connection.** §1–§4 fixed the lift, the shape, the cost, and the number format; this
section groups the operations that reuse them.

- **Rule.** The twenty sections are grouped by purpose, not alphabet.

| Family | Members | Shared reason |
|---|---|---|
| The field | $+,-,\times,/$ | affine maps and normalization |
| Quotient/remainder | $\lfloor x/y\rfloor,\ x\bmod y$ | integer index arithmetic |
| Powers and roots | $x^p,\lvert x\rvert,1/x,\sqrt{x},1/\sqrt{x}$ | norms, distances, variance |
| Transcendentals | $e^x,\ln x,2^x,\log_2 x$ | probability and information |
| Order and bounds | $\max,\min,\operatorname{clamp}$ | keeping a value inside its domain |

- **Example.** Field: affine/residual. Quotient: index decomposition. Powers/roots:
  RMSNorm. Transcendentals: log-sum-exp. Bounds: ReLU and clamp.
- **Where.** Part III walks the families in this order.
- **Triton.** All five share the skeleton; only the scalar expression and `other` change.
- **Failure.** Treating the families as unrelated hides the recurring mask, promotion, and
  domain rules.
- **Bridge.** Part II writes the scalar oracle for all twenty operations.

In [ ]:
# Why: fix the chapter spine once so every later cell uses the same literals.
# How: N=6, BLOCK=4, P=2; the second tile is partial.
N = 6
BLOCK = 4
P = (N + BLOCK - 1) // BLOCK


def spine_tensors():
    """The chapter spine: x=(1..6), y=(6..1), dtype float32."""
    x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
    y = torch.tensor([6.0, 5.0, 4.0, 3.0, 2.0, 1.0])
    return x, y


def test_tile_map():
    """Layer A — the partial last tile has two masked lanes."""
    assert P == 2
    x, y = spine_tensors()
    expected = {
        0: ([0, 1, 2, 3], [True, True, True, True], [1, 2, 3, 4], [6, 5, 4, 3]),
        1: ([4, 5, 6, 7], [True, True, False, False], [5, 6, 0, 0], [2, 1, 0, 0]),
    }
    for pid, (offs_ref, mask_ref, x_ref, y_ref) in expected.items():
        offs = pid * BLOCK + torch.arange(BLOCK)
        mask = offs < N
        safe = offs.clamp(max=N - 1)
        x_tile = torch.where(mask, x[safe], torch.zeros(BLOCK))
        y_tile = torch.where(mask, y[safe], torch.zeros(BLOCK))
        assert offs.tolist() == offs_ref
        assert mask.tolist() == mask_ref
        assert x_tile.tolist() == x_ref
        assert y_tile.tolist() == y_ref

    sweep = {
        2: (3, [4, 5], [True, True]),
        3: (2, [3, 4, 5], [True, True, True]),
        4: (2, [4, 5, 6, 7], [True, True, False, False]),
        8: (1, list(range(8)), [True] * 6 + [False, False]),
    }
    for b, (p_ref, offs_ref, mask_ref) in sweep.items():
        p = (N + b - 1) // b
        last = (p - 1) * b + torch.arange(b)
        assert p == p_ref
        assert last.tolist() == offs_ref
        assert (last < N).tolist() == mask_ref


def test_worked_example():
    """Layer A — the spine table matches the hand calculations."""
    x, y = spine_tensors()
    expected = {
        "add": torch.tensor([7.0, 7.0, 7.0, 7.0, 7.0, 7.0]),
        "sub": torch.tensor([-5.0, -3.0, -1.0, 1.0, 3.0, 5.0]),
        "mul": torch.tensor([6.0, 10.0, 12.0, 12.0, 10.0, 6.0]),
        "pow": torch.tensor([1.0, 32.0, 81.0, 64.0, 25.0, 6.0]),
        "max": torch.tensor([6.0, 5.0, 4.0, 4.0, 5.0, 6.0]),
        "clamp": torch.tensor([2.0, 2.0, 3.0, 4.0, 5.0, 5.0]),
    }
    got = {
        "add": x + y,
        "sub": x - y,
        "mul": x * y,
        "pow": torch.pow(x, y),
        "max": torch.maximum(x, y),
        "clamp": torch.clamp(x, 2.0, 5.0),
    }
    for name, ref in expected.items():
        torch.testing.assert_close(got[name], ref)


test_tile_map()
test_worked_example()


In [ ]:
# Why: traffic, not FLOPs, sets the lower bound for an elementwise kernel.
# How: the spine (N=6) and a production size (N=2^20), for b=2 and b=4.
for n in [N, 2**20]:
    for b in [2, 4]:
        unary = 2 * n * b
        binary = 3 * n * b
        assert unary == 2 * n * b
        assert binary == 3 * n * b

assert 2 * (2**20) * 4 == 8_388_608
assert 3 * (2**20) * 4 == 12_582_912
assert abs(1 / (3 * 4) - 0.0833333333) < 1e-9


### Worked numerical example (chapter spine)

**Connection.** §2 fixed the spine; this table is the reference that every later section
checks.

- **Example.** The operations that are safe on the positive spine:

| operation | result |
|---|---|
| $x+y$ | $(7,7,7,7,7,7)$ |
| $x-y$ | $(-5,-3,-1,1,3,5)$ |
| $x\,y$ | $(6,10,12,12,10,6)$ |
| $x/y$ | $(0.1667,0.4,0.75,1.3333,2.5,6)$ |
| $x^y$ | $(1,32,81,64,25,6)$ |
| $\max(x,y)$ | $(6,5,4,4,5,6)$ |
| $\min(x,y)$ | $(1,2,3,3,2,1)$ |
| $\operatorname{clamp}(x,2,5)$ | $(2,2,3,4,5,5)$ |

- **Hand check.** $z_1=2+5=7$; $2^5=32$; $\max(2,5)=5$; $\operatorname{clamp}(1,2,5)=2$.
- **Where.** `test_worked_example` verifies these rows exactly with
  `torch.equal`/`allclose`.
- **Bridge.** Part II turns each row into a scalar Python function.

## Part II — Method 01: the element-by-element reference

**Connection.** Part I defined the lift; Method 01 evaluates it one lane at a time.

- **Rule.** `method01_scalar` applies one Python operation; `method01_apply` loops over
  the vector.
- **Formula.** Binary operations read $(x_i,y_i)$; unary operations read $x_i$ and ignore
  $y$; clamp uses $[2,5]$.
- **Example.** Floor division and modulo use Python's floor semantics; the positive spine
  agrees with Triton's truncating integer operators.
- **Where.** The correctness oracle for every Triton kernel: if Method 02 disagrees with
  Method 01 on the spine, the kernel is wrong.
- **Triton.** None. This is a reference, not a baseline: a Python loop over millions of
  elements is thousands of times slower than one launch.
- **Failure.** Python floors and follows the divisor for `//` and `%`; C, CUDA, and Triton
  truncate and follow the dividend, so signed cases differ.
- **Bridge.** Part III implements the same twenty operations as tiles.

In [ ]:
def method01_scalar(op, a, b=None):
    """Method 01 — one scalar operation, written as plain Python."""
    if op == "add":
        return a + b
    if op == "sub":
        return a - b
    if op == "mul":
        return a * b
    if op == "div":
        return a / b
    if op == "floordiv":
        return math.floor(a / b)
    if op == "mod":
        return a - math.floor(a / b) * b
    if op == "pow":
        return a ** b
    if op == "neg":
        return -a
    if op == "abs":
        return abs(a)
    if op == "recip":
        return 1.0 / a
    if op == "square":
        return a * a
    if op == "sqrt":
        return math.sqrt(a)
    if op == "rsqrt":
        return 1.0 / math.sqrt(a)
    if op == "exp":
        return math.exp(a)
    if op == "log":
        return math.log(a)
    if op == "exp2":
        return 2.0 ** a
    if op == "log2":
        return math.log2(a)
    if op == "max":
        return max(a, b)
    if op == "min":
        return min(a, b)
    if op == "clamp":
        return min(max(a, 2.0), 5.0)
    raise ValueError(f"unknown op: {op}")


def method01_apply(op, x, y=None):
    """Method 01 — element-by-element reference for a vector."""
    values = []
    for i in range(len(x)):
        a = float(x[i])
        b = float(y[i]) if y is not None else None
        values.append(method01_scalar(op, a, b))
    return torch.tensor(values, dtype=torch.float32)


BINARY_OPS = {"add", "sub", "mul", "div", "floordiv", "mod", "pow", "max", "min"}


def test_pseudo_matches_torch():
    """Method 01 must match PyTorch on the spine for all twenty operations."""
    x, y = spine_tensors()
    refs = {
        "add": x + y,
        "sub": x - y,
        "mul": x * y,
        "div": x / y,
        "floordiv": torch.div(x, y, rounding_mode="floor"),
        "mod": torch.remainder(x, y),
        "pow": torch.pow(x, y),
        "neg": -x,
        "abs": (x - 3.5).abs(),
        "recip": 1.0 / x,
        "square": x * x,
        "sqrt": torch.sqrt(x),
        "rsqrt": torch.rsqrt(x),
        "exp": torch.exp(x),
        "log": torch.log(x),
        "exp2": torch.exp2(x),
        "log2": torch.log2(x),
        "max": torch.maximum(x, y),
        "min": torch.minimum(x, y),
        "clamp": torch.clamp(x, 2.0, 5.0),
    }
    for op, ref in refs.items():
        yy = y if op in BINARY_OPS else None
        xx = (x - 3.5) if op == "abs" else x
        got = method01_apply(op, xx, yy)
        torch.testing.assert_close(got, ref, atol=1e-6, rtol=1e-6)


def test_method01_worked_example():
    """The hand checks from Part I, recomputed by the scalar reference."""
    x, y = spine_tensors()
    assert method01_apply("add", x, y)[1].item() == 7.0
    assert method01_apply("sub", x, y)[1].item() == -3.0
    assert method01_apply("mul", x, y)[1].item() == 10.0
    assert method01_apply("pow", x, y)[1].item() == 32.0
    assert method01_apply("clamp", x)[0].item() == 2.0


test_pseudo_matches_torch()
test_method01_worked_example()


## Part III — Method 02: one Triton kernel per operation

**Connection.** Method 01 supplied the scalar oracle; these kernels lift it to the spine
with one expression changed.

- **Rule.** Every kernel is the same seven-line skeleton: grid $(P,)=(2,)$, block
  `BLOCK=4`, mask `offs < N`.
- **Example.** The tile map repeated because every test depends on it:

| pid | offsets | mask | live lanes |
|---|---|---|---|
| 0 | 0,1,2,3 | T,T,T,T | 0,1,2,3 |
| 1 | 4,5,6,7 | T,T,F,F | 4,5 |

- **Where.** The sections are grouped into the five families from Part I; read the family
  introduction first.
- **Triton.** `offs = pid*BLOCK + tl.arange(0, BLOCK)`; `mask = offs < N`; one expression
  per kernel.
- **Failure.** A wrong `other` can create a spurious `nan` or `inf` even on a lane that is
  never stored.
- **Bridge.** The `other` table below fixes that choice for every family.

#### Choosing `other`

- **Rule.** `other` is the value a masked lane receives from `tl.load`; it participates in
  the scalar expression even though the lane is not stored.
- **Example.** Pick a value that cannot create a spurious `nan` or `inf`:

| Following operation | Safe `other` | Why |
|---|---|---|
| addition, sum | `0.0` | additive identity |
| multiplication | `0.0` | multiplicative annihilator |
| division numerator | `0.0` | masked lane is not stored |
| division denominator | `1.0` | avoids `1/0` |
| reciprocal | `1.0` | $1/1=1$ |
| square root, rsqrt | `1.0` | $\sqrt1=1$ |
| logarithm, log2 | `1.0` | $\log1=0$ |
| exponential, exp2 | `0.0` | $e^0=2^0=1$ |
| power | `1.0` | $1^1=1$; avoids $\log_2 0$ |
| maximum (elementwise) | `0.0` | masked lane is not stored |
| minimum (elementwise) | `0.0` | masked lane is not stored |
| reduction maximum | `-inf` | cannot win against a real value |
| reduction minimum | `+inf` | cannot win against a real value |

- **Where.** Elementwise rules and reduction rules differ because a reduction actually
  reads the masked lane.
- **Triton.** The log-sum-exp kernel in Part IV uses `other=-inf`; the elementwise maximum
  kernel can use `0.0`.
- **Failure.** Reusing the elementwise choice in a reduction lets a dead lane win the
  maximum or minimum.
- **Bridge.** Family A starts with the field operations, where `other=0.0` is always safe.

## Family A — The field: $+,-,\times,/$

**Connection.** §1–§5 fixed the rules; these four sections are the first family and the
simplest lift.

- **Rule.** The field operations act lane by lane on same-shape or broadcast tiles.
- **Where.** Residual add, bias add, scale, mean subtraction, division by a norm, and
  attention scores scaled by $1/\sqrt d$.
- **Triton.** `+`, `-`, `*`, `/`; promote narrow inputs before a long sum.
- **Failure.** Subtraction can cancel, division can blow up, and multiplication can
  overflow a narrow format.
- **Bridge.** §7.1 is the neutral case; the next three flip or scale it.

### 7.1 Addition

**Connection.** §1 defined the binary lift; addition is its neutral member and the first
tile kernel.

- **Rule.** Adds two tiles lane by lane: $z_i=x_i+y_i$.
- **Formula.** $z_i=x_i+y_i$; zero is the identity.
- **Where.** Residual connections $h\leftarrow h+F(h)$, bias adds $x+b$, and accumulators
  $a\leftarrow a+\Delta$.
- **Triton.** `z = x + y`; broadcast a size-1 axis if the shapes differ.
- **Failure.** Adding two narrow types can overflow before promotion; accumulate in
  `float32` when the sum is large.
- **Bridge.** §7.2 keeps the skeleton and flips the sign of the second operand.

#### Worked example — §7.1 Addition

- **Connection.** §7.1 added two tiles; this is the same lane check the code cell
  performs.
- **Example.** On the spine $x=(1,2,3,4,5,6)$ and $y=(6,5,4,3,2,1)$,
  $z=x+y=(7,7,7,7,7,7)$.
- **Hand check.** $z_1=2+5=7$; the code prints `7.0` for lane 1.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores the same expression between the masked load
  and store.

In [ ]:
# Why: addition is the neutral combine; every lane is independent.
# How: chapter spine N=6, BLOCK=4; the second tile is partial.
x, y = spine_tensors()
z = x + y


assert torch.equal(z, torch.tensor([7.0, 7.0, 7.0, 7.0, 7.0, 7.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.1 Addition.** `m7_1_add_kernel` stores `x + y` between the same masked
load and store as every other section. The test uses the spine, so the partial second tile
is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_1_add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x + y, mask=mask)


def test_m7_1_add():
    """Method 02 — add on the chapter spine."""
    require_gpu()
    x, y = spine_tensors()
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = torch.empty_like(x)
    m7_1_add_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, x + y, atol=1e-6, rtol=1e-6)


### 7.2 Subtraction

**Connection.** §7.1 combined two tiles; subtraction keeps the same skeleton but flips the
sign of the second operand.

- **Rule.** Subtracts two tiles lane by lane: $z_i=x_i-y_i$.
- **Formula.** $z_i=x_i-y_i$; subtraction is not commutative.
- **Where.** Centering $x-\mu$, residual error $x-\hat x$, finite differences, and the max
  shift $x-\max_j x_j$ in a stable softmax.
- **Triton.** `z = x - y`.
- **Failure.** Catastrophic cancellation when $x\approx y$; compute the difference in
  `float32`.
- **Bridge.** §7.3 replaces repeated addition by a scale.

#### Worked example — §7.2 Subtraction

- **Connection.** §7.2 is addition with a flipped sign; the lane check is otherwise
  identical.
- **Example.** On the spine, $z=x-y=(-5,-3,-1,1,3,5)$.
- **Hand check.** $z_1=2-5=-3$; the code prints `-3.0` for lane 1.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `x - y` with the same mask.

In [ ]:
# Why: subtraction removes a common part or measures a difference.
# How: chapter spine N=6, BLOCK=4; lane 1 is 2 - 5.
x, y = spine_tensors()
z = x - y


assert torch.equal(z, torch.tensor([-5.0, -3.0, -1.0, 1.0, 3.0, 5.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.2 Subtraction.** `m7_2_sub_kernel` stores `x - y` between the same
masked load and store as every other section. The test uses the spine, so the partial
second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_2_sub_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x - y, mask=mask)


def test_m7_2_sub():
    """Method 02 — sub on the chapter spine."""
    require_gpu()
    x, y = spine_tensors()
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = torch.empty_like(x)
    m7_2_sub_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, x - y, atol=1e-6, rtol=1e-6)


### 7.3 Multiplication

**Connection.** §7.1–§7.2 added and subtracted; multiplication replaces repeated addition
by a scale.

- **Rule.** Multiplies two tiles lane by lane: $z_i=x_i\,y_i$.
- **Formula.** $z_i=x_i\,y_i$.
- **Where.** Scaling $a x$, gating $x\odot g$, variance $x^2$, attention scores, and the
  affine map $a x+b$.
- **Triton.** `z = x * y`; use `float32` for a sum of products.
- **Failure.** `float16` overflows above 65504; `0 * inf` is `nan`, not zero.
- **Bridge.** §7.4 inverts multiplication and introduces the denominator.

#### Worked example — §7.3 Multiplication

- **Connection.** §7.3 scales the first tile by the second; the lane check follows the
  same pattern.
- **Example.** On the spine, $z=x\,y=(6,10,12,12,10,6)$.
- **Hand check.** $z_1=2\cdot5=10$; the code prints `10.0` for lane 1.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `x * y` with the same mask.

In [ ]:
# Why: multiplication changes scale and is the product rule's forward map.
# How: chapter spine N=6, BLOCK=4; lane 1 is 2 * 5.
x, y = spine_tensors()
z = x * y


assert torch.equal(z, torch.tensor([6.0, 10.0, 12.0, 12.0, 10.0, 6.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.3 Multiplication.** `m7_3_mul_kernel` stores `x * y` between the same
masked load and store as every other section. The test uses the spine, so the partial
second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_3_mul_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * y, mask=mask)


def test_m7_3_mul():
    """Method 02 — mul on the chapter spine."""
    require_gpu()
    x, y = spine_tensors()
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = torch.empty_like(x)
    m7_3_mul_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, x * y, atol=1e-6, rtol=1e-6)


### 7.4 Division

**Connection.** §7.3 scaled by a tile; division is the inverse scale and the first
operation with a singularity.

- **Rule.** Divides two tiles lane by lane: $z_i=x_i/y_i$.
- **Formula.** $z_i=x_i/y_i$.
- **Where.** Means $\frac1N\sum_j x_j$, softmax denominators, normalized vectors
  $x/\lVert x\rVert$, and rates.
- **Triton.** `z = x / y`; the compiler may use a fast reciprocal path.
- **Failure.** A zero denominator gives $\pm\infty$ or `nan`; add an epsilon or clamp the
  denominator.
- **Bridge.** Family B moves from real quotients to integer quotient and remainder.

#### Worked example — §7.4 Division

- **Connection.** §7.4 inverts the product from §7.3; the lane check includes the
  rounding.
- **Example.** On the spine, $z=x/y=(0.1667,0.4,0.75,1.3333,2.5,6)$.
- **Hand check.** $z_1=2/5=0.4$; the code prints `0.4` for lane 1.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `x / y` with the same mask.

In [ ]:
# Why: division turns a sum into a mean and a value into a normalized value.
# How: chapter spine N=6, BLOCK=4; lane 1 is 2 / 5.
x, y = spine_tensors()
z = x / y


assert torch.allclose(z, torch.tensor([1/6, 0.4, 0.75, 4/3, 2.5, 6.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.4 Division.** `m7_4_div_kernel` stores `x / y` between the same masked
load and store as every other section. The test uses the spine, so the partial second tile
is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_4_div_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=1.0)
    tl.store(out_ptr + offs, x / y, mask=mask)


def test_m7_4_div():
    """Method 02 — div on the chapter spine."""
    require_gpu()
    x, y = spine_tensors()
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = torch.empty_like(x)
    m7_4_div_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, x / y, atol=1e-6, rtol=1e-6)


## Family B — Quotient and remainder

**Connection.** Family A produced real field values; this family keeps the same tiles but
returns integer parts.

- **Rule.** `//` answers which block a flat index belongs to; `%` answers where inside
  that block it sits.
- **Where.** Tiling, row/column decomposition, cyclic buffers, and bucketing.
- **Triton.** Integer `//` and `%`; for floats use `tl.floor(x / y)`.
- **Failure.** Python floors and follows the divisor; C, CUDA, and Triton truncate and
  follow the dividend. They agree only for non-negative indices.
- **Bridge.** §7.5 starts with the quotient; §7.6 keeps the remainder.

### 7.5 Floor division

**Connection.** Family A ended with real division; floor division truncates the quotient
to an integer.

- **Rule.** Maps a numerator to the largest integer not greater than the quotient.
- **Formula.** $z_i=\lfloor x_i/y_i\rfloor$.
- **Where.** Index decomposition `pid = flat // block`, row/column recovery, bucketing,
  and grid-stride mapping.
- **Triton.** `x // y` for integers; `tl.floor(x / y)` for floats.
- **Failure.** Triton's integer `//` truncates toward zero, unlike Python's floor;
  negative operands differ.
- **Bridge.** §7.6 recovers the remainder of the same division.

#### Worked example — §7.5 Floor division

- **Connection.** §7.5 returns the block index; this check uses the integer spine.
- **Example.** On $x=(1,2,3,4,5,6)$ and $y=(6,5,4,3,2,1)$,
  $z=x\mathbin{//}y=(0,0,0,1,2,6)$.
- **Hand check.** $6\mathbin{//}1=6$ and $1\mathbin{//}6=0$; the code prints both.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The code also prints the negative case to expose floor versus truncation.

In [ ]:
# Why: floor division maps a flat index to a block or a row.
# How: integer spine N=6, BLOCK=4; lane 5 is 6 // 1.
xi = torch.tensor([1, 2, 3, 4, 5, 6], dtype=torch.int32)
yi = torch.tensor([6, 5, 4, 3, 2, 1], dtype=torch.int32)
z = xi // yi

torch.testing.assert_close(z, torch.tensor([0, 0, 0, 1, 2, 6], dtype=torch.int32))
assert z[5].item() == 6 and z[0].item() == 0
assert -7 // 3 == -3
assert int(-7 / 3) == -2

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.5 Floor division.** `m7_5_floordiv_kernel` stores `x // y` between the
same masked load and store as every other section. The test uses the spine, so the partial
second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_5_floordiv_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0)
    y = tl.load(y_ptr + offs, mask=mask, other=1)
    tl.store(out_ptr + offs, x // y, mask=mask)


def test_m7_5_floordiv():
    """Method 02 — floordiv on the chapter spine."""
    require_gpu()
    x = torch.tensor([1, 2, 3, 4, 5, 6], device=DEVICE, dtype=torch.int32)
    y = torch.tensor([6, 5, 4, 3, 2, 1], device=DEVICE, dtype=torch.int32)
    out = torch.empty_like(x)
    m7_5_floordiv_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.div(x, y, rounding_mode="trunc"))


### 7.6 Modulo

**Connection.** §7.5 produced the quotient; modulo keeps what is left.

- **Rule.** Returns the remainder after floor division.
- **Formula.** $z_i=x_i-y_i\lfloor x_i/y_i\rfloor$.
- **Where.** Cyclic buffers, wrap-around padding, row/column recovery, and ring-buffer
  slots.
- **Triton.** `x % y` for integers; the sign follows the dividend (C semantics).
- **Failure.** Python's `%` follows the divisor; Triton's follows the dividend. Normalize
  before wrapping.
- **Bridge.** Family C moves from indices to magnitudes.

#### Worked example — §7.6 Modulo

- **Connection.** §7.6 is the remainder left by §7.5's quotient.
- **Example.** On the integer spine, $z=x\bmod y=(1,2,3,1,1,0)$.
- **Hand check.** $5\bmod2=1$ and $6\bmod1=0$; the code prints the pair table.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The code also prints the negative case to expose divisor versus dividend
  sign.

In [ ]:
# Why: modulo gives the position inside a block or a cyclic buffer.
# How: integer spine N=6, BLOCK=4; lane 5 is 6 % 1.
xi = torch.tensor([1, 2, 3, 4, 5, 6], dtype=torch.int32)
yi = torch.tensor([6, 5, 4, 3, 2, 1], dtype=torch.int32)
z = xi % yi

torch.testing.assert_close(z, torch.tensor([1, 2, 3, 1, 1, 0], dtype=torch.int32))
assert -7 % 3 == 2
assert -7 - int(-7 / 3) * 3 == -1

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.6 Modulo.** `m7_6_mod_kernel` stores `x % y` between the same masked
load and store as every other section. The test uses the spine, so the partial second tile
is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_6_mod_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0)
    y = tl.load(y_ptr + offs, mask=mask, other=1)
    tl.store(out_ptr + offs, x % y, mask=mask)


def test_m7_6_mod():
    """Method 02 — mod on the chapter spine."""
    require_gpu()
    x = torch.tensor([1, 2, 3, 4, 5, 6], device=DEVICE, dtype=torch.int32)
    y = torch.tensor([6, 5, 4, 3, 2, 1], device=DEVICE, dtype=torch.int32)
    out = torch.empty_like(x)
    m7_6_mod_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, x - torch.div(x, y, rounding_mode="trunc") * y)


## Family C — Powers and roots

**Connection.** Families A–B covered field values and integer indices; Family C produces
magnitudes, distances, and norms.

- **Rule.** A norm is a sum of squares followed by a root; a variance is a mean of squares
  minus a squared mean; a normalization divides by a root or a reciprocal root.
- **Where.** Negation and absolute value belong to the same geometric story: a distance is
  a magnitude, and a direction is a sign.
- **Triton.** `x*x`, `tl.sqrt`, `tl.rsqrt`, `tl.abs`, and `tl.exp2(y * tl.log2(x))` for a
  positive-base power.
- **Failure.** Domain and range: a square root of a negative number is not real, a
  reciprocal of zero is not finite, and a real power of a negative base is not real unless
  the exponent is an integer.
- **Bridge.** §7.7 generalizes multiplication to a power.

### 7.7 Power

**Connection.** §7.3 multiplied two tiles; power repeats that product with a tile
exponent.

- **Rule.** Raises a base to an exponent: $x^y=2^{y\log_2 x}$ for $x\gt 0$.
- **Formula.** $z_i=x_i^{\,y_i}$.
- **Where.** Variance, polynomial features, distances $\lVert x\rVert^p$, and scaling
  laws.
- **Triton.** `tl.exp2(y * tl.log2(x))` for a positive base; `libdevice.pow(x, y)` for a
  general real power.
- **Failure.** A negative base with a fractional exponent is `nan`; Triton 3.x has no
  tensor `**` operator.
- **Bridge.** §7.8 is the sign flip that power and distance arguments need.

#### Worked example — §7.7 Power

- **Connection.** §7.7 generalizes §7.3; the positive spine avoids the negative-base
  branch.
- **Example.** On the spine, $z=x^{y}=(1,32,81,64,25,6)$.
- **Hand check.** $2^5=32$ and $3^4=81$; the code prints both.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The code also verifies that `exp2(y*log2(x))` matches `torch.pow` on the
  spine.

In [ ]:
# Why: power is repeated multiplication or exp2(y * log2(x)).
# How: positive spine N=6, BLOCK=4; lane 1 is 2 ** 5.
x, y = spine_tensors()
z = torch.pow(x, y)
z_log2 = torch.exp2(y * torch.log2(x))

torch.testing.assert_close(z, torch.tensor([1.0, 32.0, 81.0, 64.0, 25.0, 6.0]))
torch.testing.assert_close(z_log2, z, atol=1e-4, rtol=1e-4)

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.7 Power.** `m7_7_pow_kernel` stores `tl.exp2(y * tl.log2(x))` between
the same masked load and store as every other section. The positive-base route runs on the
interpreter; the general `libdevice.pow` route is GPU-only. The GPU test is collected at
the end of Part III.

In [ ]:
@triton.jit
def m7_7_pow_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=1.0)
    y = tl.load(y_ptr + offs, mask=mask, other=1.0)
    # positive-base route; use libdevice.pow(x, y) for a signed base
    tl.store(out_ptr + offs, tl.exp2(y * tl.log2(x)), mask=mask)


def test_m7_7_pow():
    """Method 02 — pow on the chapter spine."""
    require_gpu()
    x, y = spine_tensors()
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = torch.empty_like(x)
    m7_7_pow_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.pow(x, y), atol=1e-4, rtol=1e-4)


### 7.8 Negation

**Connection.** §7.2 subtracted; negation is subtraction from zero and the additive
inverse.

- **Rule.** Flips the sign of every lane: $z_i=-x_i$.
- **Formula.** $z_i=-x_i$.
- **Where.** Descent directions $-\nabla L$, removing a term $a-b=a+(-b)$, and symmetric
  windows.
- **Triton.** `z = -x`.
- **Failure.** Negating the most negative integer overflows; `-0.0` and `0.0` compare
  equal but divide differently.
- **Bridge.** §7.9 drops the sign and keeps the magnitude.

#### Worked example — §7.8 Negation

- **Connection.** §7.8 is the unary form of §7.2 with $y=0$; this check uses the positive
  spine.
- **Example.** On the spine, $z=-x=(-1,-2,-3,-4,-5,-6)$.
- **Hand check.** $-2=-2$; the code prints `-2.0` for lane 1.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below is the first unary kernel in the chapter.

In [ ]:
# Why: negation is the additive inverse and the descent direction.
# How: chapter spine N=6, BLOCK=4; lane 1 is -2.
x, _ = spine_tensors()
z = -x


assert torch.equal(z, torch.tensor([-1.0, -2.0, -3.0, -4.0, -5.0, -6.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.8 Negation.** `m7_8_neg_kernel` stores `-x` between the same masked
load and store as every other section. The test uses the spine, so the partial second tile
is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_8_neg_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, -x, mask=mask)


def test_m7_8_neg():
    """Method 02 — neg on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_8_neg_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, -x, atol=1e-6, rtol=1e-6)


### 7.9 Absolute value

**Connection.** §7.8 flipped the sign; absolute value keeps the magnitude and discards the
direction.

- **Rule.** Removes the sign: $z_i=\lvert x_i\rvert$.
- **Formula.** $z_i=\lvert x_i\rvert$.
- **Where.** Distances $\lvert x-y\rvert$, L1 norms, error metrics, and the stable
  softplus $\max(x,0)+\log(1+e^{-\lvert x\rvert})$.
- **Triton.** `z = tl.abs(x)`.
- **Failure.** `abs` of the most negative integer overflows; the derivative at zero is not
  unique.
- **Bridge.** §7.10 inverts a value, the first reciprocal of the family.

#### Worked example — §7.9 Absolute value

- **Connection.** §7.9 is the magnitude left after §7.8 removes the sign; the spine is
  shifted so both signs appear.
- **Example.** On $s=x-3.5=(-2.5,-1.5,-0.5,0.5,1.5,2.5)$,
  $\lvert s\rvert=(2.5,1.5,0.5,0.5,1.5,2.5)$.
- **Hand check.** $\lvert 2-3.5\rvert=1.5$; the code prints `1.5` for lane 1.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.abs(x)` with the same mask.

In [ ]:
# Why: absolute value is magnitude without direction.
# How: shift the spine by 3.5 so both signs appear.
x, _ = spine_tensors()
s = x - 3.5
z = s.abs()


assert torch.equal(z, torch.tensor([2.5, 1.5, 0.5, 0.5, 1.5, 2.5]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.9 Absolute value.** `m7_9_abs_kernel` stores `tl.abs(x)` between the
same masked load and store as every other section. The test uses the spine, so the partial
second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_9_abs_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, tl.abs(x), mask=mask)


def test_m7_9_abs():
    """Method 02 — abs on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x - 3.5).to(DEVICE)
    out = torch.empty_like(x)
    m7_9_abs_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, x.abs(), atol=1e-6, rtol=1e-6)


### 7.10 Reciprocal

**Connection.** §7.4 divided by a tile; reciprocal is division by a constant one and the
inverse scale.

- **Rule.** Returns the multiplicative inverse: $z_i=1/x_i$.
- **Formula.** $z_i=1/x_i$.
- **Where.** Normalization, odds $p/(1-p)$, learning-rate schedules, and division-free
  forms.
- **Triton.** `z = 1.0 / x`.
- **Failure.** Zero gives $\pm\infty$; the reciprocal of a tiny value can overflow.
- **Bridge.** §7.11 squares; with §7.12 it builds norms.

#### Worked example — §7.10 Reciprocal

- **Connection.** §7.10 is §7.4 with a constant denominator of one; this check uses the
  positive spine.
- **Example.** On the spine, $1/x=(1,0.5,0.3333,0.25,0.2,0.1667)$.
- **Hand check.** $1/4=0.25$; the code prints `0.25` for lane 3.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `1.0 / x` with the same mask.

In [ ]:
# Why: the reciprocal is division without a division.
# How: positive spine N=6, BLOCK=4; lane 3 is 1 / 4.
x, _ = spine_tensors()
z = 1.0 / x


assert torch.allclose(z, torch.tensor([1.0, 0.5, 1/3, 0.25, 0.2, 1/6]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.10 Reciprocal.** `m7_10_recip_kernel` stores `1.0 / x` between the
same masked load and store as every other section. The test uses the spine, so the partial
second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_10_recip_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=1.0)
    tl.store(out_ptr + offs, 1.0 / x, mask=mask)


def test_m7_10_recip():
    """Method 02 — recip on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_10_recip_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, 1.0 / x, atol=1e-6, rtol=1e-6)


### 7.11 Square

**Connection.** §7.3 multiplied two tiles; square is the special case $y=x$ and the atom
of variance.

- **Rule.** Squares every lane: $z_i=x_i^2$.
- **Formula.** $z_i=x_i^2$.
- **Where.** Variance $\mathbb E[x^2]-\mathbb E[x]^2$, energy, and squared Euclidean
  distance.
- **Triton.** `z = x * x`.
- **Failure.** Overflow in narrow formats: $256^2$ is outside `float16`; accumulate
  squares in `float32`.
- **Bridge.** §7.12 undoes the square.

#### Worked example — §7.11 Square

- **Connection.** §7.11 is §7.3 with both operands equal; this check uses the positive
  spine.
- **Example.** On the spine, $x^2=(1,4,9,16,25,36)$.
- **Hand check.** $4^2=16$; the code prints `16.0` for lane 3.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `x * x` with the same mask.

In [ ]:
# Why: the square is the atom of variance and distance.
# How: chapter spine N=6, BLOCK=4; lane 3 is 4 * 4.
x, _ = spine_tensors()
z = x * x


assert torch.equal(z, torch.tensor([1.0, 4.0, 9.0, 16.0, 25.0, 36.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.11 Square.** `m7_11_square_kernel` stores `x * x` between the same
masked load and store as every other section. The test uses the spine, so the partial
second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_11_square_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * x, mask=mask)


def test_m7_11_square():
    """Method 02 — square on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_11_square_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, x * x, atol=1e-6, rtol=1e-6)


### 7.12 Square root

**Connection.** §7.11 squared; square root returns the non-negative magnitude.

- **Rule.** Takes the non-negative square root: $z_i=\sqrt{x_i}$.
- **Formula.** $z_i=\sqrt{x_i}$.
- **Where.** Euclidean norms, standard deviations, RMS normalization, and the attention
  scale $1/\sqrt d$.
- **Triton.** `z = tl.sqrt(x)`; `tl.sqrt_rn(x)` is the correctly rounded `float32` form.
- **Failure.** A negative input is `nan`; the derivative is infinite at zero.
- **Bridge.** §7.13 combines §7.10 and §7.12.

#### Worked example — §7.12 Square root

- **Connection.** §7.12 inverts §7.11; this check uses the positive spine.
- **Example.** On the spine, $\sqrt x=(1,1.4142,1.7321,2,2.2361,2.4495)$.
- **Hand check.** $\sqrt4=2$; the code prints `2.0` for lane 3.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.sqrt(x)` with the same mask.

In [ ]:
# Why: the square root turns a squared norm into a norm.
# How: positive spine N=6, BLOCK=4; lane 3 is sqrt(4).
x, _ = spine_tensors()
z = torch.sqrt(x)

torch.testing.assert_close(
    z,
    torch.tensor([1.0, 2**0.5, 3**0.5, 2.0, 5**0.5, 6**0.5]),
    atol=1e-6,
    rtol=1e-6,
)
assert z[3].item() == 2.0

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.12 Square root.** `m7_12_sqrt_kernel` stores `tl.sqrt(x)` between the
same masked load and store as every other section. The test uses the spine, so the partial
second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_12_sqrt_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=1.0)
    tl.store(out_ptr + offs, tl.sqrt(x), mask=mask)


def test_m7_12_sqrt():
    """Method 02 — sqrt on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_12_sqrt_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.sqrt(x), atol=1e-6, rtol=1e-6)


### 7.13 Reciprocal square root

**Connection.** §7.10 inverted and §7.12 took a root; `rsqrt` does both in one operation.

- **Rule.** Combines the inverse and the square root: $z_i=x_i^{-1/2}$.
- **Formula.** $z_i=1/\sqrt{x_i}$.
- **Where.** RMSNorm $x\cdot\operatorname{rsqrt}(\mathbb E[x^2]+\epsilon)$, LayerNorm, and
  the attention scale.
- **Triton.** `z = tl.rsqrt(x)`.
- **Failure.** Zero gives `inf`; a negative input gives `nan`; epsilon placement matters.
- **Bridge.** Family D exponentiates to move from additive logits to multiplicative
  weights.

#### Worked example — §7.13 Reciprocal square root

- **Connection.** §7.13 composes §7.10 and §7.12; this check uses the positive spine.
- **Example.** On the spine,
  $\operatorname{rsqrt}(x)=(1,0.7071,0.5774,0.5,0.4472,0.4082)$.
- **Hand check.** $1/\sqrt4=0.5$; the code prints `0.5` for lane 3.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.rsqrt(x)` with the same mask.

In [ ]:
# Why: rsqrt is the normalization operation of RMSNorm and LayerNorm.
# How: positive spine N=6, BLOCK=4; lane 3 is rsqrt(4).
x, _ = spine_tensors()
z = torch.rsqrt(x)

torch.testing.assert_close(
    z,
    torch.tensor([1.0, 0.7071, 0.5774, 0.5, 0.4472, 0.4082]),
    atol=1e-4,
    rtol=1e-4,
)
assert z[3].item() == 0.5

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.13 Reciprocal square root.** `m7_13_rsqrt_kernel` stores `tl.rsqrt(x)`
between the same masked load and store as every other section. The test uses the spine, so
the partial second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_13_rsqrt_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=1.0)
    tl.store(out_ptr + offs, tl.rsqrt(x), mask=mask)


def test_m7_13_rsqrt():
    """Method 02 — rsqrt on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_13_rsqrt_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.rsqrt(x), atol=1e-4, rtol=1e-4)


## Family D — Transcendentals

**Connection.** Family C produced magnitudes and roots; Family D is the bridge between
additive logits and multiplicative probabilities.

- **Rule.** The exponential turns one into the other, and the logarithm turns it back. The
  natural pair $(e^x,\ln x)$ is the calculus pair; the base-2 pair $(2^x,\log_2 x)$ is the
  hardware-native pair.
- **Formula.** $e^x=2^{x\log_2 e}$ and $\ln x=\log_2 x\cdot\ln2$.
- **Where.** Softmax, sigmoid, softplus, cross-entropy, and log-sum-exp all live here.
- **Triton.** `tl.exp`/`tl.log` for the natural pair; `tl.exp2`/`tl.log2` for the fast
  pair. Convert the constant, not the operation.
- **Failure.** Range: `exp` overflows at about 88.7 in `float32`, and `log` is undefined
  at zero and negative inputs.
- **Bridge.** §7.14 starts with the exponential.

### 7.14 Exponential

**Connection.** §7.13 produced an inverse root; §7.14 is the first transcendental and maps
addition to multiplication.

- **Rule.** Raises $e$ to each lane: $z_i=e^{x_i}$.
- **Formula.** $z_i=e^{x_i}$.
- **Where.** Softmax numerators, sigmoid, softplus, decay $e^{-\lambda t}$, and Poisson
  weights.
- **Triton.** `z = tl.exp(x)`.
- **Failure.** Overflows above about 88.7 in `float32`; subtract the row maximum first.
- **Bridge.** §7.15 is its inverse.

#### Worked example — §7.14 Exponential

- **Connection.** §7.14 introduces the natural exponential; this check uses the positive
  spine.
- **Example.** On the spine, $e^x=(2.7183,7.3891,20.0855,54.5982,148.4132,403.4288)$.
- **Hand check.** $e^1=2.71828$; the code prints that value for lane 0.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.exp(x)` with the same mask.

In [ ]:
# Why: exp turns additive logits into multiplicative weights.
# How: spine N=6, BLOCK=4; lane 0 is e ** 1.
x, _ = spine_tensors()
z = torch.exp(x)

torch.testing.assert_close(
    z,
    torch.tensor([2.7183, 7.3891, 20.0855, 54.5982, 148.4132, 403.4288]),
    atol=1e-4,
    rtol=1e-4,
)
assert abs(z[0].item() - math.e) < 1e-6

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.14 Exponential.** `m7_14_exp_kernel` stores `tl.exp(x)` between the
same masked load and store as every other section. The test uses the spine, so the partial
second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_14_exp_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, tl.exp(x), mask=mask)


def test_m7_14_exp():
    """Method 02 — exp on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_14_exp_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.exp(x), atol=1e-5, rtol=1e-5)


### 7.15 Natural logarithm

**Connection.** §7.14 turned sums into products; the logarithm turns products back into
sums.

- **Rule.** Takes the natural logarithm: $z_i=\ln x_i$.
- **Formula.** $z_i=\ln x_i$.
- **Where.** Cross-entropy, log-likelihood, entropy, and the log-sum-exp.
- **Triton.** `z = tl.log(x)`.
- **Failure.** $\ln0=-\infty$; a negative input is `nan`; do not take the log of an
  underflowed probability.
- **Bridge.** §7.16 switches to the hardware-native base.

#### Worked example — §7.15 Natural logarithm

- **Connection.** §7.15 inverts §7.14; this check uses the positive spine.
- **Example.** On the spine, $\ln x=(0,0.6931,1.0986,1.3863,1.6094,1.7918)$.
- **Hand check.** $\ln1=0$ and $\ln4=1.38629$; the code prints both.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.log(x)` with the same mask.

In [ ]:
# Why: log turns multiplicative weights back into additive quantities.
# How: positive spine N=6, BLOCK=4; lane 0 is ln(1).
x, _ = spine_tensors()
z = torch.log(x)

torch.testing.assert_close(
    z,
    torch.tensor([0.0, 0.6931, 1.0986, 1.3863, 1.6094, 1.7918]),
    atol=1e-4,
    rtol=1e-4,
)
assert abs(z[0].item()) < 1e-6
assert abs(z[3].item() - math.log(4)) < 1e-6

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.15 Natural logarithm.** `m7_15_log_kernel` stores `tl.log(x)` between
the same masked load and store as every other section. The test uses the spine, so the
partial second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_15_log_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=1.0)
    tl.store(out_ptr + offs, tl.log(x), mask=mask)


def test_m7_15_log():
    """Method 02 — log on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_15_log_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.log(x), atol=1e-6, rtol=1e-6)


### 7.16 Base-2 exponential

**Connection.** §7.14 used base $e$; `exp2` is the instruction hardware actually
implements and the route to $e^x$ and $x^y$.

- **Rule.** Raises two to each lane: $z_i=2^{x_i}$.
- **Formula.** $z_i=2^{x_i}$; $e^x=2^{x\log_2 e}$.
- **Where.** The hardware-native exponential, powers of two, and positive-base powers.
- **Triton.** `z = tl.exp2(x)`.
- **Failure.** Overflows above about 128 in `float32`; base confusion with `tl.exp`.
- **Bridge.** §7.17 is its inverse.

#### Worked example — §7.16 Base-2 exponential

- **Connection.** §7.16 is the base-2 form of §7.14; this check uses the positive spine.
- **Example.** On the spine, $2^x=(2,4,8,16,32,64)$.
- **Hand check.** $2^3=8$; the code prints `8.0` for lane 2.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.exp2(x)` with the same mask.

In [ ]:
# Why: exp2 is the instruction the hardware actually implements.
# How: spine N=6, BLOCK=4; lane 2 is 2 ** 3.
x, _ = spine_tensors()
z = torch.exp2(x)


assert torch.equal(z, torch.tensor([2.0, 4.0, 8.0, 16.0, 32.0, 64.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.16 Base-2 exponential.** `m7_16_exp2_kernel` stores `tl.exp2(x)`
between the same masked load and store as every other section. The test uses the spine, so
the partial second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_16_exp2_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, tl.exp2(x), mask=mask)


def test_m7_16_exp2():
    """Method 02 — exp2 on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_16_exp2_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.exp2(x), atol=1e-5, rtol=1e-5)


### 7.17 Base-2 logarithm

**Connection.** §7.15 used base $e$; `log2` is the hardware-native logarithm and the
inverse of §7.16.

- **Rule.** Takes the base-2 logarithm: $z_i=\log_2 x_i$.
- **Formula.** $z_i=\log_2 x_i$; $\ln x=\log_2 x\cdot\ln2$.
- **Where.** Information measured in bits, binary entropy, and the hardware-native
  logarithm.
- **Triton.** `z = tl.log2(x)`.
- **Failure.** $\log_2 0=-\infty$; a negative input is `nan`; base confusion with
  `tl.log`.
- **Bridge.** Family E closes the chapter by keeping values inside a set.

#### Worked example — §7.17 Base-2 logarithm

- **Connection.** §7.17 inverts §7.16; this check uses the positive spine.
- **Example.** On the spine, $\log_2 x=(0,1,1.5850,2,2.3219,2.5850)$.
- **Hand check.** $\log_2 4=2$; the code prints `2.0` for lane 3.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.log2(x)` with the same mask.

In [ ]:
# Why: log2 measures information in bits and is the hardware-native log.
# How: positive spine N=6, BLOCK=4; lane 3 is log2(4).
x, _ = spine_tensors()
z = torch.log2(x)

torch.testing.assert_close(
    z,
    torch.tensor([0.0, 1.0, 1.5850, 2.0, 2.3219, 2.5850]),
    atol=1e-4,
    rtol=1e-4,
)
assert z[3].item() == 2.0

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.17 Base-2 logarithm.** `m7_17_log2_kernel` stores `tl.log2(x)` between
the same masked load and store as every other section. The test uses the spine, so the
partial second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_17_log2_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=1.0)
    tl.store(out_ptr + offs, tl.log2(x), mask=mask)


def test_m7_17_log2():
    """Method 02 — log2 on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = (x).to(DEVICE)
    out = torch.empty_like(x)
    m7_17_log2_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.log2(x), atol=1e-6, rtol=1e-6)


## Family E — Order and bounds

**Connection.** Family D produced unbounded exponentials; Family E bounds values and
guards the other families.

- **Rule.** A maximum enforces a lower bound, a minimum enforces an upper bound, and a
  clamp enforces both.
- **Where.** ReLU, ReLU6, hardtanh, gradient clipping, and the guards in front of `log`,
  `sqrt`, and `rsqrt`.
- **Triton.** `tl.maximum(x, y)` and `tl.minimum(x, y)` are elementwise;
  `tl.max(x, axis=...)` and `tl.min(x, axis=...)` are reductions.
- **Failure.** Two traps: the elementwise/reduction name collision, and NaN policy.
  Triton's elementwise `maximum`/`minimum` default to `propagate_nan=NONE`, which ignores
  NaN; PyTorch's `torch.maximum` propagates it.
- **Bridge.** §7.18 starts with the lower bound.

### 7.18 Maximum

**Connection.** §7.17 ended the transcendental pair; maximum begins the bound family with
a lower bound.

- **Rule.** Returns the larger of two lanes: $z_i=\max(x_i,y_i)$.
- **Formula.** $z_i=\max(x_i,y_i)$.
- **Where.** ReLU, the max shift in stable softmax, max-pooling, and lower bounds.
- **Triton.** `tl.maximum(x, y)` is elementwise; `tl.max(x, axis=...)` is a reduction.
- **Failure.** Triton's default `propagate_nan=NONE` ignores NaN; `torch.maximum`
  propagates it.
- **Bridge.** §7.19 is the symmetric upper bound.

#### Worked example — §7.18 Maximum

- **Connection.** §7.18 enforces the lower bound; this check compares two full tiles.
- **Example.** On the spine, $\max(x,y)=(6,5,4,4,5,6)$.
- **Hand check.** $\max(2,5)=5$; the code prints `5.0` for lane 1.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.maximum(x, y)` with the same mask.

In [ ]:
# Why: elementwise maximum is a lower bound and a winner.
# How: chapter spine N=6, BLOCK=4; lane 1 is max(2, 5).
x, y = spine_tensors()
z = torch.maximum(x, y)


assert torch.equal(z, torch.tensor([6.0, 5.0, 4.0, 4.0, 5.0, 6.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.18 Maximum.** `m7_18_maximum_kernel` stores `tl.maximum(x, y)` between
the same masked load and store as every other section. The test uses the spine, so the
partial second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_18_maximum_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, tl.maximum(x, y), mask=mask)


def test_m7_18_maximum():
    """Method 02 — maximum on the chapter spine."""
    require_gpu()
    x, y = spine_tensors()
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = torch.empty_like(x)
    m7_18_maximum_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.maximum(x, y), atol=1e-6, rtol=1e-6)


### 7.19 Minimum

**Connection.** §7.18 kept the larger lane; minimum keeps the smaller and enforces an
upper bound.

- **Rule.** Returns the smaller of two lanes: $z_i=\min(x_i,y_i)$.
- **Formula.** $z_i=\min(x_i,y_i)$.
- **Where.** Upper bounds, min-pooling, hardtanh, and denominator guards.
- **Triton.** `tl.minimum(x, y)` is elementwise; `tl.min(x, axis=...)` is a reduction.
- **Failure.** The same NaN and broadcasting issues as `maximum`; do not confuse the
  elementwise and reduction forms.
- **Bridge.** §7.20 uses both bounds at once.

#### Worked example — §7.19 Minimum

- **Connection.** §7.19 enforces the upper bound; this check compares two full tiles.
- **Example.** On the spine, $\min(x,y)=(1,2,3,3,2,1)$.
- **Hand check.** $\min(2,5)=2$; the code prints `2.0` for lane 1.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.minimum(x, y)` with the same mask.

In [ ]:
# Why: elementwise minimum is an upper bound and a guard.
# How: chapter spine N=6, BLOCK=4; lane 1 is min(2, 5).
x, y = spine_tensors()
z = torch.minimum(x, y)


assert torch.equal(z, torch.tensor([1.0, 2.0, 3.0, 3.0, 2.0, 1.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.19 Minimum.** `m7_19_minimum_kernel` stores `tl.minimum(x, y)` between
the same masked load and store as every other section. The test uses the spine, so the
partial second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_19_minimum_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, tl.minimum(x, y), mask=mask)


def test_m7_19_minimum():
    """Method 02 — minimum on the chapter spine."""
    require_gpu()
    x, y = spine_tensors()
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = torch.empty_like(x)
    m7_19_minimum_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.minimum(x, y), atol=1e-6, rtol=1e-6)


### 7.20 Clamping

**Connection.** §7.18 and §7.19 supplied the two one-sided bounds; clamping composes them.

- **Rule.** Keeps every lane inside $[l,u]$: $z_i=\min(\max(x_i,l),u)$.
- **Formula.** $z_i=\min(\max(x_i,l),u)$ with $l\le u$.
- **Where.** ReLU6, hardtanh, gradient clipping, and the guards in front of `log`, `sqrt`,
  and `rsqrt`.
- **Triton.** `tl.clamp(x, lo, hi)`; `lo <= hi` is required.
- **Failure.** `lo > hi` is undefined; clamping a value is not clamping its gradient; NaN
  policy differs from torch.
- **Bridge.** Part IV fuses these twenty operations into chains.

#### Worked example — §7.20 Clamping

- **Connection.** §7.20 composes §7.18 and §7.19; this check uses $l=2$ and $u=5$.
- **Example.** On the spine, $\operatorname{clamp}(x,2,5)=(2,2,3,4,5,5)$.
- **Hand check.** $\operatorname{clamp}(1,2,5)=2$ and $\operatorname{clamp}(6,2,5)=5$; the
  code prints both.
- **Steps.** Read the named lane, apply the scalar formula, compare with the print.
- **Bridge.** The micro-kernel below stores `tl.clamp(x, lo, hi)` with the same mask.

In [ ]:
# Why: clamping is the guard that makes the other families safe.
# How: chapter spine N=6, BLOCK=4; lo=2, hi=5.
x, _ = spine_tensors()
z = torch.clamp(x, 2.0, 5.0)


assert torch.equal(z, torch.tensor([2.0, 2.0, 3.0, 4.0, 5.0, 5.0]))

# Knob: change BLOCK from 4 to 3 and re-run; the values do not move,
# only the tile boundary does.


**Micro-kernel — §7.20 Clamping.** `m7_20_clamp_kernel` stores `tl.clamp(x, lo, hi)`
between the same masked load and store as every other section. The test uses the spine, so
the partial second tile is exercised; the GPU test is collected at the end of Part III.

In [ ]:
@triton.jit
def m7_20_clamp_kernel(x_ptr, out_ptr, n_elements, lo, hi, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    z = tl.clamp(x, lo, hi)
    tl.store(out_ptr + offs, z, mask=mask)


def test_m7_20_clamp():
    """Method 02 — clamp on the chapter spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = x.to(DEVICE)
    out = torch.empty_like(x)
    m7_20_clamp_kernel[(P,)](x, out, N, 2.0, 5.0, BLOCK=BLOCK)
    torch.testing.assert_close(out, x.clamp(2.0, 5.0))


### 7.21 The canonical elementwise kernel

**Connection.** §7.1–§7.20 changed only the scalar expression; this section collects the
invariant they share and hands it to Part IV.

- **Rule.** Every kernel in the chapter is the same seven-line skeleton: `pid`, `offs`,
  `mask`, masked load, scalar body, masked store.
- **Formula.** $\text{offs}=\text{pid}\cdot B+\{0,\dots,B-1\}$,
  $\text{mask}=\text{offs}\lt N$, and $P=\lceil N/B\rceil$.
- **Example.** One kernel with `OP: tl.constexpr` covers add, sub, mul, div, max, and min;
  the JIT emits only the selected arm, so the unused arms cost nothing at runtime.
- **Where.** This is the bridge to the kernel part: comparisons (Chapter 8), activations
  (Chapter 9), broadcasting (Chapter 10), and fused chains (Chapter 20) reuse the skeleton.
- **Triton.** The canonical form, with one `other` choice that is safe for every arm:

```python
@triton.jit
def canonical_elementwise_kernel(x_ptr, y_ptr, out_ptr, n_elements,
                                 OP: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=1.0)
    if OP == 0:
        z = x + y
    elif OP == 1:
        z = x - y
    elif OP == 2:
        z = x * y
    elif OP == 3:
        z = x / y
    elif OP == 4:
        z = tl.maximum(x, y)
    else:
        z = tl.minimum(x, y)
    tl.store(out_ptr + offs, z, mask=mask)
```

- **Failure.** A runtime `OP` would branch per lane and add divergence; keep the selector
  `tl.constexpr`. The denominator uses `other=1.0` because division is the one arm that is
  not safe at zero.
- **Bridge.** Part IV changes the grid and fuses several bodies, but the body remains this
  skeleton.


### Run the Method 02 tests

**Connection.** §7.1–§7.20 defined one kernel each; §7.21 collected their shared skeleton;
this loop runs the twenty kernels against PyTorch on the chapter spine.

- **Rule.** On a GPU every test runs; on a CPU-only machine each test prints `SKIP ...`.
- **Where.** `METHOD02_TESTS` collects all twenty entries in section order.
- **Failure.** A skipped test is not a pass; the numerical cells above are what still run
  on CPU.
- **Bridge.** The Layer C check below re-verifies the ladder end to end.

In [ ]:
METHOD02_TESTS = [
    test_m7_1_add,
    test_m7_2_sub,
    test_m7_3_mul,
    test_m7_4_div,
    test_m7_5_floordiv,
    test_m7_6_mod,
    test_m7_7_pow,
    test_m7_8_neg,
    test_m7_9_abs,
    test_m7_10_recip,
    test_m7_11_square,
    test_m7_12_sqrt,
    test_m7_13_rsqrt,
    test_m7_14_exp,
    test_m7_15_log,
    test_m7_16_exp2,
    test_m7_17_log2,
    test_m7_18_maximum,
    test_m7_19_minimum,
    test_m7_20_clamp,
]

for test in METHOD02_TESTS:
    run_if_gpu(test)


**Connection.** The loop checks each kernel separately; this check re-verifies the Method
02 add kernel against the Method 01 spine.

- **Rule.** Run `m7_1_add_kernel` on the spine and compare with `x + y` using
  `torch.equal`.
- **Example.** The same call is wrapped as `test_correctness` in the next code cell.
- **Where.** Layer C of the method ladder: kernel output versus the scalar oracle.

```python
x, y = spine_tensors()
x, y = x.to(DEVICE), y.to(DEVICE)
out = torch.empty_like(x)
m7_1_add_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
assert torch.equal(out, x + y)
print("method 02 matches spine: True")
```

- **Bridge.** Part IV composes the kernels and reduces traffic.

In [ ]:
def test_correctness():
    """Method 02 — the add kernel reproduces the spine after the ladder."""
    require_gpu()
    x, y = spine_tensors()
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = torch.empty_like(x)
    m7_1_add_kernel[(P,)](x, y, out, N, BLOCK=BLOCK)
    assert torch.equal(out, x + y)


# The book's Method ladder calls this check `test_correctness`.
test_method02_spine = test_correctness
run_if_gpu(test_method02_spine)


## Part IV — Method 03: fused and production-shaped kernels

**Connection.** Method 02 launched one kernel per operation; Method 03 composes them so
intermediates stay in registers.

- **Rule.** Fusion changes traffic, not the calculus: load once, evaluate the chain, store
  once.
- **Where.** A fused arithmetic chain, four applied kernels, the softmax capstone, and the
  backward derivation.
- **Triton.** One kernel holds several scalar expressions; a compile-time mode selects a
  branch without runtime divergence.
- **Failure.** A fused kernel that recomputes a value is still cheaper than a round trip
  through HBM, but fusion does not remove a singularity or a domain error.
- **Bridge.** The fused chain is the smallest example of the pattern.

## Fused arithmetic chain

**Connection.** Method 02's add, multiply, subtract, and divide kernels are the pieces;
this section composes them into one launch.

- **Rule.** Compose $y_i=((x_i+2)\cdot3-x_i)/2$, which simplifies to $x_i+3$.
- **Formula.** $Q_{\text{fused}}=2Nb$ instead of $Q_{\text{unfused}}=4\cdot2Nb$.
- **Example.** The unfused version launches four kernels and writes three intermediate
  vectors; the fused version loads $x_i$ once and stores $y_i$ once.
- **Where.** The extra identity term `abs(-x) - abs(x)` exercises absolute value and
  negation; a production kernel would drop it.
- **Triton.** Evaluate the whole expression in registers between one masked load and one
  masked store.
- **Failure.** Do not fuse a chain whose intermediate is needed elsewhere; that would
  recompute or lose a value.
- **Bridge.** Applied examples 1–4 instantiate the same fusion pattern.

In [ ]:
@triton.jit
def m3_fused_chain_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    y = ((x + 2.0) * 3.0 - x) / 2.0
    y = y + tl.abs(-x) - tl.abs(x)   # identity; exercises abs and negation

    tl.store(out_ptr + offs, y, mask=mask)


def method03_reference(x):
    """Method 01/03 reference: the unfused expression, evaluated in PyTorch."""
    return ((x + 2.0) * 3.0 - x) / 2.0


def test_method03_reference():
    """Method 03 — the unfused reference simplifies to x + 3."""
    x, _ = spine_tensors()
    ref = method03_reference(x)
    assert torch.allclose(ref, x + 3.0)


def test_method03_fused():
    """Method 03 — the fused chain matches the unfused reference."""
    require_gpu()
    x, _ = spine_tensors()
    x = x.to(DEVICE)
    out = torch.empty_like(x)
    m3_fused_chain_kernel[(P,)](x, out, N, BLOCK=BLOCK)
    torch.testing.assert_close(out, method03_reference(x), atol=1e-6, rtol=1e-6)


test_method03_reference()
run_if_gpu(test_method03_fused)


## Applied example 1 — Affine transform and residual

**Connection.** The fused chain showed one expression; the affine map is the smallest
useful composition.

- **Rule.** $y_i=a\,x_i+b$ is one multiplication and one addition.
- **Formula.** The residual form is $y_i=x_i+a'\,x_i=(1+a')\,x_i$, the same map with $b=0$
  and $a=1+a'$.
- **Example.** On the spine with $a=2$, $b=-1$, $y=(1,3,5,7,9,11)$; with $a=1.5$, $b=0$,
  $y=(1.5,3,4.5,6,7.5,9)$.
- **Where.** The per-channel affine of a normalization layer, the scale and shift of a
  residual branch, and the linear part of a small MLP.
- **Triton.** `z = a * x + b` between one masked load and one masked store.
- **Failure.** A narrow `a` or `b` can overflow before promotion; compute the affine in
  `float32`.
- **Bridge.** Applied 2 uses the same map inside a normalization.

In [ ]:
x, _ = spine_tensors()
torch.testing.assert_close(2.0 * x - 1.0, torch.tensor([1.0, 3.0, 5.0, 7.0, 9.0, 11.0]))
torch.testing.assert_close(x + 0.5 * x, torch.tensor([1.5, 3.0, 4.5, 6.0, 7.5, 9.0]))


@triton.jit
def affine_residual_kernel(x_ptr, out_ptr, n_elements, a, b, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    z = a * x + b
    tl.store(out_ptr + offs, z, mask=mask)


def test_affine_residual():
    """Applied 1 — affine and residual on the spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = x.to(DEVICE)
    out = torch.empty_like(x)
    affine_residual_kernel[(P,)](x, out, N, 2.0, -1.0, BLOCK=BLOCK)
    torch.testing.assert_close(out, 2.0 * x - 1.0)

    affine_residual_kernel[(P,)](x, out, N, 1.5, 0.0, BLOCK=BLOCK)
    torch.testing.assert_close(out, x + 0.5 * x)


## Applied example 2 — RMS normalization

**Connection.** Applied 1 built an affine map; RMSNorm uses the root mean square as the
scale.

- **Rule.**
  $y_i=x_i\cdot\operatorname{rsqrt}\left(\frac1N\sum_{j=1}^N x_j^2+\epsilon\right)$.
- **Formula.** It composes square, sum, division, addition, rsqrt, and multiplication;
  LayerNorm adds a mean subtraction and an affine map, while RMSNorm drops both.
- **Example.** On the spine, $\frac16\sum_j x_j^2=91/6=15.1667$, so the inverse RMS is
  $\operatorname{rsqrt}(15.1667+10^{-6})=0.2568$ and
  $y\approx(0.2568,0.5136,0.7704,1.0272,1.2840,1.5408)$.
- **Where.** RMSNorm and LayerNorm.
- **Triton.** One row per program: load the row, reduce with `tl.sum`, apply `tl.rsqrt`,
  store the scaled row.
- **Failure.** Without epsilon a row of zeros produces $\operatorname{rsqrt}(0)=\infty$
  and then $0\cdot\infty=\text{nan}$.
- **Bridge.** Applied 3 needs a stable way to normalize a sum of exponentials.

In [ ]:
x, _ = spine_tensors()
eps = 1e-6
mean_sq = (x * x).mean()
inv_rms = torch.rsqrt(mean_sq + eps)
ref = x * inv_rms

assert abs(mean_sq.item() - 91 / 6) < 1e-6
assert abs(inv_rms.item() - 0.2568) < 1e-4
torch.testing.assert_close(
    ref,
    torch.tensor([0.2568, 0.5136, 0.7704, 1.0272, 1.2840, 1.5408]),
    atol=1e-4,
    rtol=1e-4,
)

BLOCK_NORM = triton.next_power_of_2(N)   # one row per program; 8 for N=6


@triton.jit
def rms_norm_kernel(x_ptr, out_ptr, n_elements, eps, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    mean_sq = tl.sum(x * x, axis=0) / n_elements
    inv_rms = tl.rsqrt(mean_sq + eps)
    tl.store(out_ptr + offs, x * inv_rms, mask=mask)


def test_rms_norm():
    """Applied 2 — RMS normalization on the spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = x.to(DEVICE)
    out = torch.empty_like(x)
    rms_norm_kernel[(1,)](x, out, N, eps, BLOCK=BLOCK_NORM)
    torch.testing.assert_close(
        out, x * torch.rsqrt((x * x).mean() + eps), atol=1e-5, rtol=1e-5
    )


## Applied example 3 — Stable log-sum-exp

**Connection.** Applied 2 normalized by a root mean square; log-sum-exp is the stable
normalization of a sum of exponentials.

- **Rule.** $\operatorname{LSE}(x)=\log\sum_{j=1}^N e^{x_j}$.
- **Formula.** The stable form is $\operatorname{LSE}(x)=m+\log\sum_{j=1}^N e^{x_j-m}$
  with $m=\max_j x_j$.
- **Example.** On the spine, $m=6$ and
  $\operatorname{LSE}(x)=6+\log(e^{-5}+e^{-4}+e^{-3}+e^{-2}+e^{-1}+1)\approx6.4562$.
- **Where.** Softmax denominators, cross-entropy, and log-domain arithmetic.
- **Triton.** `other=-inf` for the masked maximum, then `tl.max`, `tl.exp`, `tl.sum`, and
  `tl.log`.
- **Failure.** The direct form overflows above about 88.7 in `float32`; `other=0.0` could
  win the maximum when all real logits are negative.
- **Bridge.** Applied 4 uses bounds as guards.

In [ ]:
x, _ = spine_tensors()
ref = torch.logsumexp(x, dim=0)

assert abs(ref.item() - 6.456193) < 1e-5
torch.testing.assert_close(
    torch.exp(x),
    torch.tensor([2.7183, 7.3891, 20.0855, 54.5982, 148.4132, 403.4288]),
    atol=1e-4,
    rtol=1e-4,
)


@triton.jit
def logsumexp_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=-float("inf"))
    m = tl.max(x, axis=0)
    lse = m + tl.log(tl.sum(tl.exp(x - m), axis=0))
    tl.store(out_ptr, lse)


def test_logsumexp():
    """Applied 3 — stable log-sum-exp on the spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = x.to(DEVICE)
    out = torch.empty(1, device=x.device, dtype=x.dtype)
    logsumexp_kernel[(1,)](x, out, N, BLOCK=BLOCK_NORM)
    torch.testing.assert_close(out[0], torch.logsumexp(x, dim=0), atol=1e-5, rtol=1e-5)


## Applied example 4 — Bounds: ReLU, hardtanh, safe log, safe sqrt

**Connection.** Applied 3 used `other=-inf` as a neutral element; bounds supply the finite
guards.

- **Rule.** Each mode is one or two bounds followed by an operation:

| Mode | Formula | Use |
|---|---|---|
| 0 | $\max(x,0)$ | ReLU |
| 1 | $\min(\max(x,-1),1)$ | hardtanh |
| 2 | $\log(\max(x,\epsilon))$ | safe logarithm |
| 3 | $\sqrt{\max(x,0)}$ | safe square root |

- **Example.** On the spine with $l=-1$ and $u=1$, $\operatorname{ReLU}(x)=(1,2,3,4,5,6)$
  and $\operatorname{hardtanh}(x)=(1,1,1,1,1,1)$.
- **Where.** A host branch would be wrong because the condition is per lane; a device
  branch would diverge; a clamp is a branch-free selection that every lane executes.
- **Triton.** `tl.maximum`, `tl.clamp`, `tl.log`, and `tl.sqrt` inside one compile-time
  mode selector.
- **Failure.** A guard applied after the dangerous operation hides the overflow instead of
  preventing it.
- **Bridge.** The capstone combines all five families.

In [ ]:
x, _ = spine_tensors()
torch.testing.assert_close(torch.relu(x), x)
torch.testing.assert_close(x.clamp(-1.0, 1.0), torch.ones_like(x))
assert torch.isfinite(torch.log(torch.clamp(x, min=1e-6))).all()
assert torch.isfinite(torch.sqrt(torch.relu(x))).all()

BOUND_RELU, BOUND_HARDTANH, BOUND_LOG, BOUND_SQRT = range(4)


@triton.jit
def bounds_kernel(x_ptr, out_ptr, n_elements, MODE: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    if MODE == BOUND_RELU:
        z = tl.maximum(x, 0.0)
    elif MODE == BOUND_HARDTANH:
        z = tl.clamp(x, -1.0, 1.0)
    elif MODE == BOUND_LOG:
        z = tl.log(tl.maximum(x, 1e-6))
    else:
        z = tl.sqrt(tl.maximum(x, 0.0))

    tl.store(out_ptr + offs, z, mask=mask)


def test_bounds():
    """Applied 4 — ReLU, hardtanh, safe log, safe sqrt."""
    require_gpu()
    x, _ = spine_tensors()
    x = x.to(DEVICE)
    out = torch.empty_like(x)

    bounds_kernel[(P,)](x, out, N, MODE=BOUND_RELU, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.relu(x))

    bounds_kernel[(P,)](x, out, N, MODE=BOUND_HARDTANH, BLOCK=BLOCK)
    torch.testing.assert_close(out, x.clamp(-1.0, 1.0))

    bounds_kernel[(P,)](x, out, N, MODE=BOUND_LOG, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.log(torch.clamp(x, min=1e-6)))

    bounds_kernel[(P,)](x, out, N, MODE=BOUND_SQRT, BLOCK=BLOCK)
    torch.testing.assert_close(out, torch.sqrt(torch.relu(x)))


## Capstone — a stable softmax from arithmetic

**Connection.** Applied 1–4 used the pieces; softmax uses every family at once.

- **Rule.** $\operatorname{softmax}(x)_i=e^{x_i-m}/\sum_{j=1}^N e^{x_j-m}$ with
  $m=\max_j x_j$.
- **Formula.** The maximum is a reduction (`tl.max`), the shift is subtraction, the
  exponential is `tl.exp`, the denominator is a sum, and the final division is `/`.
- **Example.** On the spine the reference probabilities are approximately
  $(0.0043,0.0116,0.0315,0.0858,0.2331,0.6337)$.
- **Where.** The result is a probability vector: non-negative entries that sum to one.
- **Triton.** One program per row; a production softmax would process many rows per launch
  and autotune the block size.
- **Failure.** Without the shift, a logit of 100 overflows before the division; with it,
  the largest exponent is $e^0=1$.
- **Bridge.** Part V differentiates the same chain.

In [ ]:
x, _ = spine_tensors()
ref = torch.softmax(x, dim=0)

torch.testing.assert_close(
    ref,
    torch.tensor([0.0043, 0.0116, 0.0315, 0.0858, 0.2331, 0.6337]),
    atol=1e-4,
    rtol=1e-4,
)
assert abs(ref.sum().item() - 1.0) < 1e-6


@triton.jit
def softmax_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=-float("inf"))
    m = tl.max(x, axis=0)
    e = tl.exp(x - m)
    s = tl.sum(e, axis=0)
    tl.store(out_ptr + offs, e / s, mask=mask)


def test_softmax():
    """Capstone — stable softmax on the spine."""
    require_gpu()
    x, _ = spine_tensors()
    x = x.to(DEVICE)
    out = torch.empty_like(x)
    softmax_kernel[(1,)](x, out, N, BLOCK=BLOCK_NORM)
    torch.testing.assert_close(out, torch.softmax(x, dim=0), atol=1e-6, rtol=1e-6)


## Part V — Forward and backward

**Connection.** Part IV composed forwards; this part differentiates them and shows that
the backward is another elementwise kernel.

- **Rule.** Every operation has a local derivative; the backward of a composition
  multiplies those derivatives along each path.
- **Formula.** For a binary forward, $\partial L/\partial x=g\,\partial z/\partial x$ and
  $\partial L/\partial y=g\,\partial z/\partial y$; a broadcast axis becomes a sum in the
  backward.
- **Example.** The local derivatives used in the chapter:

| Operation | Forward | Local derivative |
|---|---|---|
| addition | $x+y$ | $\partial_x=1,\ \partial_y=1$ |
| subtraction | $x-y$ | $\partial_x=1,\ \partial_y=-1$ |
| multiplication | $xy$ | $\partial_x=y,\ \partial_y=x$ |
| division | $x/y$ | $\partial_x=1/y,\ \partial_y=-x/y^2$ |
| power | $x^p$ | $p x^{p-1}$ |
| negation | $-x$ | $-1$ |
| absolute value | $\lvert x\rvert$ | $\operatorname{sign}(x)$ |
| reciprocal | $1/x$ | $-1/x^2$ |
| square | $x^2$ | $2x$ |
| square root | $\sqrt{x}$ | $1/(2\sqrt{x})$ |
| reciprocal square root | $x^{-1/2}$ | $-1/(2x^{3/2})$ |
| exponential | $e^x$ | $e^x$ |
| natural logarithm | $\ln x$ | $1/x$ |
| base-2 exponential | $2^x$ | $2^x\ln2$ |
| base-2 logarithm | $\log_2 x$ | $1/(x\ln2)$ |
| maximum | $\max(x,y)$ | $\mathbf 1[x\gt y],\ \mathbf 1[y\gt x]$ |
| minimum | $\min(x,y)$ | $\mathbf 1[x\lt y],\ \mathbf 1[y\lt x]$ |
| clamp | $\min(\max(x,l),u)$ | $\mathbf 1[l\lt x\lt u]$ |

- **Where.** The derivatives of floor, modulo, maximum, minimum, and clamp are piecewise
  constant or undefined at the joins; the gradient is a convention (a subgradient), not a
  classical derivative.
- **Triton.** The backward kernel has the same shape, mask, and traffic, plus one extra
  input for $g$; the derivative is just another scalar expression.
- **Failure.** A wrong gradient usually shows up as a constant factor or a missing term;
  the table is the checklist.
- **Bridge.** The worked chain below uses two entries from the table.

## Worked chain on the spine

**Connection.** The table is general; this chain uses the logarithm from §7.15 and the
reciprocal from §7.10.

- **Rule.** Take the row loss $L(x)=\sum_{i=1}^N \ln x_i$.
- **Formula.** The forward value is $L=\ln1+\ln2+\cdots+\ln6=\ln720\approx6.5793$; the
  derivative is $\partial L/\partial x_i=1/x_i$.
- **Example.** The gradient on the spine is $(1,0.5,0.3333,0.25,0.2,0.1667)$.
- **Where.** This is the simplest chain that uses two families.
- **Triton.** The autograd cell compares `x.grad` with `1/x`.
- **Failure.** For a longer chain, multiply every local derivative along the path and add
  the contributions from every path.
- **Bridge.** The next code cell checks the printout.

In [ ]:
x, _ = spine_tensors()
x = x.clone().requires_grad_(True)

loss = torch.log(x).sum()
loss.backward()

expected = 1.0 / x.detach()
assert abs(loss.item() - math.log(720)) < 1e-6
torch.testing.assert_close(x.grad, expected, atol=1e-6, rtol=1e-6)
assert (x.grad - expected).abs().max().item() < 1e-6


**Connection.** The code above computed the log-sum forward and its autograd gradient;
this note states what the printout should show.

- **Rule.** For $L=\sum_i\ln x_i$, the only local derivative is $1/x_i$.
- **Example.** The forward value is $\ln720\approx6.5793$; the autograd gradient is
  $(1,0.5,0.3333,0.25,0.2,0.1667)$; the maximum error is at the level of floating-point
  rounding.
- **Where.** The derivative table above is the checklist.
- **Failure.** A gradient wrong by a factor is usually a missing local derivative; a
  longer chain multiplies every local derivative along the path.
- **Bridge.** The next section explains why the backward kernel has the forward's shape.

## Why the backward of an elementwise kernel is elementwise

**Connection.** The chain rule produced a scalar derivative per lane; this section reads
that derivative as a kernel.

- **Rule.** The forward reads a tile, applies $f$, and writes a tile; the backward reads
  $g=\partial L/\partial z$, applies $f'$, and writes $g\,f'(x)$.
- **Formula.** Same shape, same mask, same traffic as the forward, plus the upstream
  gradient.
- **Example.** A fused forward can be paired with a fused backward without a new memory
  layout.
- **Where.** A binary operation produces two gradients, one per input.
- **Triton.** Recompute $f'(x)$ from the saved input or load a saved mask; recomputing is
  usually cheaper for a memory-bound kernel.
- **Failure.** If the forward broadcast a size-1 axis, the backward must sum over that
  axis; the bias gradient is a reduction even when the forward looks elementwise.
- **Bridge.** That reduction is the Chapter 10 story.

## Pitfalls

**Connection.** Part V closed the calculus; these are the cross-cutting traps that survive
every family.

- **Mask both doors.** A masked store with an unmasked load still performs an
  out-of-bounds read. The mask belongs on `tl.load` and `tl.store`.
- **Choose `other` from the next operation.** `0.0` is safe for addition and sums, but a
  maximum wants `-inf`, a minimum wants `+inf`, and a logarithm must not be masked to
  zero.
- **Do not pass a `y_ptr` to a unary kernel.** Negation, square, sqrt, exp, and log have
  one input. An unused argument suggests a binary operation that does not exist.
- **Respect the domain.** `log(0)=-inf`, `log(-1)=nan`, `sqrt(-1)=nan`, `rsqrt(0)=inf`,
  and a negative base with a fractional exponent is `nan`.
- **Mind the exponent range.** `exp` overflows above about 88.7 and `exp2` above about 128
  in `float32`; `float16` products overflow above 65504.
- **Use base-2 for the hardware path.** `tl.exp2`/`tl.log2` are the fast pair;
  `tl.exp`/`tl.log` are the natural pair. Convert the constant, not the operation. For a
  general real power use `libdevice.pow(x, y)`; Triton 3.x has no tensor `**`.
- **Do not confuse elementwise and reduction.** `tl.maximum(x,y)` compares two tiles;
  `tl.max(x, axis=...)` collapses an axis. The same holds for `minimum`/`min`. Triton's
  elementwise `maximum`/`minimum` default to `propagate_nan=NONE`, while `torch.maximum`
  propagates NaN.
- **Integer division signs.** Triton's `//` truncates toward zero and `%` follows the
  dividend; Python floors and follows the divisor. For non-negative indices they agree.

## Self-check

1. For the spine $N=6$, `BLOCK=4`, write the offsets, mask, and live lanes of program 1.
   Which two lanes are masked?
2. Compute $x+y$, $x\,y$, and $x^y$ for lanes 1 and 5 by hand.
3. Why is $2^5=32$ but `exp2(5*log2(-2))` not a real number?
4. What is the difference between `tl.maximum(x,y)` and `tl.max(x, axis=0)`?
5. For $x=-7$, $y=3$, what are $x\mathbin{//}y$ and $x\bmod y$ in Python, in C, and in
   Triton?
6. Why does the stable log-sum-exp subtract $\max_j x_j$ before the exponential?
7. Derive $d/dx\,\operatorname{softplus}(x)$ and identify the sigmoid.
8. Why does `other=0.0` work for the addition kernel but not for the maximum in
   log-sum-exp?

## Summary

- **One structure.** Twenty elementwise operations: load a tile, apply a scalar function,
  store the tile.
- **Five families.** The field builds affine maps and normalization; quotient/remainder
  builds index arithmetic; powers and roots build norms, distances, and variance;
  transcendentals build probability and information; order and bounds keep a value inside
  its domain.
- **One skeleton.** The kernel skeleton is always the same; the choice of `other` is not,
  because it must be the neutral element of the operation that follows.

## Key takeaways

- Elementwise arithmetic is memory-bound. The lever is bytes, not instructions.
- The tile shape never changes; only the scalar expression does.
- A domain is part of the API. `log`, `sqrt`, `rsqrt`, division, and real powers all have
  restricted domains.
- Use `tl.exp2` / `tl.log2` for the fast exponential and for positive-base powers; use
  `libdevice.pow` for a general real power.
- `tl.maximum` / `tl.minimum` are elementwise; `tl.max` / `tl.min` are reductions.
- Triton's integer `//` and `%` truncate toward zero; Python floors. For non-negative
  indices they agree.
- The backward pass of an elementwise map is another elementwise map.

## Checkpoints

- [ ] Why is a binary elementwise kernel memory-bound even though it performs one FLOP per
  element?
- [ ] Why is `other=0.0` safe for an addition kernel but wrong for a log-sum-exp maximum?
- [ ] What is the difference between `tl.maximum(x, y)` and `tl.max(x, axis=0)`?
- [ ] Why does Triton need `exp2(y * log2(x))` or `libdevice.pow` instead of `x ** y` for
  two tiles?
- [ ] For $x=-7$, $y=3$, what are `x // y` and `x % y` in Python, in C, and in Triton?
- [ ] Derive the gradient of softplus and explain where the sigmoid comes from.

## Exercises

1. **Unary skeleton.** Rewrite `m7_1_add_kernel` for $z=\lvert x\rvert$. Remove the
   `y_ptr` argument. Explain why removing it is a correctness and readability fix, not a
   performance optimization.
2. **Floor versus truncate.** For $x=-7$, $y=3$, print Python's `x // y` and `x % y`, then
   print Triton's integer results. Write a helper that produces floor semantics from
   Triton's truncating operators.
3. **Three ways to cube.** Compute $x^3$ three ways: `x * x * x`,
   `tl.exp2(3.0 * tl.log2(x))` for positive $x$, and `libdevice.pow(x, 3.0)`. Compare
   accuracy and range.
4. **Base-2 exponential.** Replace `tl.exp(x)` with `tl.exp2(x * 1.4426950408889634)`.
   Measure the maximum relative error on $[-10,10]$ and on $[-80,80]$.
5. **Stable softplus.** Implement $\max(x,0)+\log(1+e^{-\lvert x\rvert})$ and compare with
   the naive $\log(1+e^x)$ on $[-100,100]$.
6. **Safe normalization.** Write an RMSNorm kernel with `rsqrt(var + eps)` and one with
   `rsqrt(max(var, eps))`. Feed a row of zeros and a row of very small values. Which one
   is finite? Which one is closer to PyTorch?
7. **NaN policy.** Compare `tl.maximum(x, y)` with and without
   `propagate_nan=tl.PropagateNan.ALL` on a tile containing `nan`. Which one matches
   `torch.maximum`?
8. **Backward of rsqrt.** Derive $d/dx\,x^{-1/2}$ and check it numerically against
   autograd for $x\gt 0$.
9. **Traffic accounting.** For a chain of three binary elementwise operations, compute the
   traffic of the unfused version and the fused version. How many bytes does fusion save
   for $N=2^{20}$ and $b=4$?
10. **One family kernel.** Extend the binary skeleton with an `OP: tl.constexpr` selector
    covering add, sub, mul, div, max, and min. Explain why the unused arms cost nothing at
    runtime.

Hints are in [`../../docs/SOLUTIONS.md`](../../docs/SOLUTIONS.md).

Next: [Chapter 8 — Comparisons and Logical
Operations](../ch08_comparisons/08_complete.ipynb)

[Index](../../00_index.ipynb) · [← Previous](../../part2_memory/ch06_indexing/06_complete.ipynb) · [Next →](../ch08_comparisons/08_complete.ipynb)

## Run everything

**Connection.** The chapter ends with one CPU pass and one GPU pass over the same spine.

- **Rule.** CPU tests run directly; GPU tests are collected in one loop.
- **Where.** Without a GPU each GPU test prints `SKIP <name>: ...` and the notebook
  continues.
- **Failure.** A failing check stops the cell; a skip does not.
- **Bridge.** The printed `done` line is the chapter's final check.

In [ ]:
CPU_TESTS = [
    test_tile_map,
    test_worked_example,
    test_pseudo_matches_torch,
    test_method01_worked_example,
    test_method03_reference,
]

GPU_TESTS = METHOD02_TESTS + [
    test_method02_spine,
    test_method03_fused,
    test_affine_residual,
    test_rms_norm,
    test_logsumexp,
    test_bounds,
    test_softmax,
]

print("CPU tests")
for test in CPU_TESTS:
    test()

print()
print("GPU tests")
for test in GPU_TESTS:
    run_if_gpu(test)

print()
print("done: CPU checks passed; GPU checks ran or printed SKIP")
